In [6]:
import os
import json
import time
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm

def main():
    # --- Check GPU Status ---
    print("--- GPU / CUDA Device Check ---")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    if device.type == 'cuda':
        print("GPU Name:", torch.cuda.get_device_name(0))
        print("CUDA Capability:", torch.cuda.get_device_capability(0))
        print("Total Memory:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print("[WARNING] CUDA GPU not found. Training will run on CPU (slower).")

    # Offline-First Dataset Resolver
    cache_dir = os.path.expanduser(r"~\.cache\kagglehub\datasets\tongpython\cat-and-dog\versions\1")
    if os.path.exists(cache_dir):
        path = cache_dir
        print("\n[INFO] Found cached dataset locally at:", path)
    else:
        print("\n[INFO] Attempting to download dataset from Kaggle...")
        try:
            path = kagglehub.dataset_download("tongpython/cat-and-dog")
            print("Path to dataset files:", path)
        except Exception as e:
            print("[ERROR] Connection to Kaggle failed. If offline, make sure the dataset is placed at ~/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1")
            raise e

    train_dir = os.path.join(path, "training_set", "training_set")
    test_dir = os.path.join(path, "test_set", "test_set")

    # ==========================================
    # 2. DATA PIPELINE SETUP
    # ==========================================
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 32

    data_transforms = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    print("\n--- Loading Datasets ---")
    full_train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms)
    print(f"Loaded {len(full_train_dataset)} training images belonging to classes: {full_train_dataset.classes}")

    val_size = int(0.2 * len(full_train_dataset))
    train_size = len(full_train_dataset) - val_size

    train_dataset, val_dataset = random_split(
        full_train_dataset, 
        [train_size, val_size], 
        generator=torch.Generator().manual_seed(42)
    )

    test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms)
    print(f"Loaded {len(test_dataset)} test images.")

    # num_workers=0 on Windows prevents lockups
    num_workers = 0
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)

    # ==========================================
    # 3. VGG16 MODEL ARCHITECTURE
    # ==========================================
    print("\n--- Loading Pretrained VGG16 Model ---")
    vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

    # Freeze weights
    for param in vgg16.features.parameters():
        param.requires_grad = False

    # Customize top layers
    vgg16.classifier = nn.Sequential(
        nn.Linear(25088, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 1),
        nn.Sigmoid()
    )

    model = vgg16.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.classifier.parameters(), lr=0.0001)

    # ==========================================
    # 4. TRAINING WITH PROGRESS BARS
    # ==========================================
    print("\n--- Starting Training ---")
    EPOCHS = 5
    history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}

    for epoch in range(EPOCHS):
        epoch_start = time.time()
        
        # Training loop
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
        for inputs, labels in train_bar:
            inputs = inputs.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            preds = (outputs >= 0.5).float()
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)
            
            train_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct_train/total_train:.4f}")
            
        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct_train / total_train
        
        # Validation loop
        model.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)
        with torch.no_grad():
            for inputs, labels in val_bar:
                inputs = inputs.to(device)
                labels = labels.to(device).float().unsqueeze(1)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item() * inputs.size(0)
                preds = (outputs >= 0.5).float()
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)
                
                val_bar.set_postfix(val_loss=f"{loss.item():.4f}", val_acc=f"{correct_val/total_val:.4f}")
                
        epoch_val_loss = running_val_loss / len(val_dataset)
        epoch_val_acc = correct_val / total_val
        
        history["loss"].append(epoch_loss)
        history["accuracy"].append(epoch_acc)
        history["val_loss"].append(epoch_val_loss)
        history["val_accuracy"].append(epoch_val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} [{time.time() - epoch_start:.1f}s] - "
              f"loss: {epoch_loss:.4f} - acc: {epoch_acc:.4f} - "
              f"val_loss: {epoch_val_loss:.4f} - val_acc: {epoch_val_acc:.4f}")

    # ==========================================
    # 5. EVALUATION
    # ==========================================
    print("\n--- Evaluating on Test Dataset ---")
    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0

    test_bar = tqdm(test_loader, desc="Testing Model", leave=False)
    with torch.no_grad():
        for inputs, labels in test_bar:
            inputs = inputs.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item() * inputs.size(0)
            preds = (outputs >= 0.5).float()
            correct_test += (preds == labels).sum().item()
            total_test += labels.size(0)

    final_test_loss = test_loss / len(test_dataset)
    final_test_acc = correct_test / total_test
    print(f"Final Test Accuracy: {final_test_acc * 100:.2f}%")

    # ==========================================
    # 7. SAVE MODEL & TRAINING HISTORY
    # ==========================================
    MODEL_SAVE_PATH = "cat_dog_vgg16_pytorch.pth"
    torch.save(model, MODEL_SAVE_PATH)
    print(f"\n[SUCCESS] PyTorch Model saved to: {MODEL_SAVE_PATH}")

    HISTORY_SAVE_PATH = "training_history.json"
    history_serializable = {k: [float(val) for val in v] for k, v in history.items()}
    with open(HISTORY_SAVE_PATH, "w") as f:
        json.dump(history_serializable, f, indent=2)
    print(f"[SUCCESS] Training history saved to: {HISTORY_SAVE_PATH}")

if __name__ == "__main__":
    main()


--- GPU / CUDA Device Check ---
Using device: cuda
GPU Name: NVIDIA GeForce RTX 3070 Laptop GPU
CUDA Capability: (8, 6)
Total Memory: 8.59 GB

[INFO] Found cached dataset locally at: C:\Users\Khaled Tahawy\.cache\kagglehub\datasets\tongpython\cat-and-dog\versions\1

--- Loading Datasets ---
Loaded 8005 training images belonging to classes: ['cats', 'dogs']
Loaded 2023 test images.

--- Loading Pretrained VGG16 Model ---

--- Starting Training ---


Epoch 1/5 [45.9s] - loss: 0.0610 - acc: 0.9761 - val_loss: 0.0502 - val_acc: 0.9794


Epoch 2/5 [47.0s] - loss: 0.0097 - acc: 0.9964 - val_loss: 0.0491 - val_acc: 0.9844


Epoch 3/5 [47.7s] - loss: 0.0032 - acc: 0.9992 - val_loss: 0.0511 - val_acc: 0.9825


Epoch 4/5 [46.5s] - loss: 0.0012 - acc: 0.9998 - val_loss: 0.0474 - val_acc: 0.9863


Epoch 5/5 [46.5s] - loss: 0.0007 - acc: 1.0000 - val_loss: 0.0498 - val_acc: 0.9856

--- Evaluating on Test Dataset ---


Final Test Accuracy: 98.62%

[SUCCESS] PyTorch Model saved to: cat_dog_vgg16_pytorch.pth
[SUCCESS] Training history saved to: training_history.json
